# 冲击集中度 × 自身回撤：风险价值双因子检验

冻结来源、决策时钟、同状态物理动作、预算与停止门见 `plan.json`。本卡只测局部风险和价值传递，不运行完整三资产账户。

In [1]:
%run research/stage1_shock_cushion_risk_joint_20261110/run.py

In [2]:
import json
from pathlib import Path
print(json.loads(Path('research/stage1_shock_cushion_risk_joint_20261110/summary.json').read_text())['result'])

{'status': 'closed_fixed_shock_cushion_risk_interaction_failed', 'train_fit_days': 303, 'train_fit_rows': 15150, 'test_held_source_days': 248, 'test_held_known_days': 247, 'control_dayweighted_risk_mse': 0.001006729573641999, 'candidate_dayweighted_risk_mse': 0.0009311801691664471, 'trainmean_dayweighted_risk_mse': 0.017054326839259618, 'mse_improvement_block_lower': 3.4259443987504095e-07, 'mse_undefined_bootstrap': 0, 'interaction_standardized_coefficient': 0.02642003885432014, 'information_gate_passed': True, 'changed_action_days': 0, 'same_model_zero_changed_action_days': 0, 'candidate_selected_unknown_anchors': 5, 'risk_improvement_mean': 0.0, 'risk_improvement_block_lower': 0.0, 'value_damage_mean': 0.0, 'value_damage_block_upper': 0.0, 'action_undefined_bootstrap': 0, 'value_undefined_bootstrap': 0, 'action_gate_passed': False, 'model_fits': 2, 'fit_calls_in_this_attempt': 2, 'previous_postfit_lost_model_calls': 2, 'actual_solver_invocations_total': 4, 'candidate_accounts': 0, '

# 冲击集中度 × 自身回撤：风险信息通过，行动未传递

**SOXX 判定：历史费用后稳定超额未取得。** 这是一项现有数据、同状态物理动作的风险信息筛选，未运行连续三资产完整账户或 0/2/5bp SOXX 门。

前一完整交易日 SOXX 21 日绝对冲击集中度，与决策时已接收自身回撤相乘；对照同时含两者主效应、十项自身状态、两项动作和报价费率。Qlib 来源覆盖 2018–2020 的 558 日；pi2 保存的未来路径风险标签在 2020 年 247/248 日可用。“历史风险已知”另有不同含义，不作为本次过滤。2018–2019 的 303 日、15150 个已成熟拟合动作训练；2020 的 248 日、三种预留物理动作/锚点评价。

候选 2020 按日等权风险 MSE **0.00093118**，低于加性对照 **0.00100673** 和训练日均常数 **0.01705433**；对照减候选的 10 会话区块下界 **+3.43e−7**，标准化交互系数 **+0.02642**，方向符合冻结假说。因此这个同状态局部**风险信息门通过**。但是三种预留整股行动的最小预测风险排序在 248 日内 **0 日改变**；候选交互置零也 0 日改变，5 个选中锚点的结果未知。冻结行动门失败，局部风险改善和费用后价值增量均为零。固定三行动排序版本关闭，不调符号、门限或动作集救援。该门只对这三种 held 行动与这个选择规则有效；没有同窗 Q2 风险 critic 比较或完整 actor/account 验证。

实现收据：首次 DatasetH 分组设置错误在拟合前停止；第二次两次拟合完成后在索引名对齐处失败，没有产生科学评分/Recorder；精确修复后重放同一冻结计划，保存检查点再完成两次拟合。因此实际求解器调用合计 **4** 次，最终模型 **2** 个，没有隐藏消耗。原始错误与修复范围见 `attempt1_failure.json`、`attempt2_failure.json` 和 `repair_plan.json`。

原生 Qlib Recorder `be415a07e7024c9483790172823d4900` NAS 归档 SHA `07582f2ba6e56ae6ccd4684b25c714cdb6bcb17df0f2e07848d75124be71e232`；限定恢复读取 result、predictions、action_rows、candidate_model 通过。这个因子只能保留为条件风险信息线索；要成为配置方案，还须有事前冻结的风险与费用后价值共同目标、真正改变的合法行动、与既有 Q2 critic 的匹配增量，以及自身完整账户 SOXX 门。


作用域补充：The pi2 state/policy that generated these saved action labels was itself trained using 2018-2020 development data. Held denotes held physical action IDs, not a market out-of-sample policy. The 2020 risk information pass is conditional development evidence only.


**行动接口纠正（2026-11-11，只读，0拟合）：**保存预测的同模型交互置零差在 1240 个锚点的三动作内几乎完全相同，最大组内跨度 1.11e−16。因此原交互只给三动作风险分数共同加一项，按最小风险排序必然 0 天改变。这证明固定行动门设计无法检验该因子的动作边际作用；不能据 0 天变化否定潜在的“来源×动作风险”机制，也不能把风险 MSE 改善直接称为 SOXX 超额。旧版本保持关闭，另卡需事先冻结动作相关交互及匹配 Q2/价值控制。



**后续来源机械预检（2026-11-11，0拟合）：**候选改为前日冲击集中度×已接收自身回撤×动作名义 beta 增量 ‘3u_SOXL+u_SOXX’。在包含十项状态、动作、状态×动作、动作二次项、费用和冲击主效应的来源控制后，2018–2019 训练源残差方差占原候选方差 1.95%；2020 年 1240 个预留动作锚点有 1230 个在三动作间实际变化。它解决“共同平移”的机械不可辨识性，但尚未读取新目标、拟合、比较 Q2 critic 或运行账户；仍是 `training_ready=false`。详见 `action_coupled_source_preflight.json`。



**固定30%风险预算可达性预检（2026-11-12，只读，0标签收益读取/拟合/账户）：**状态项共同平移虽不能改变同锚点最小风险排序，理论上可以让绝对风险分数跨过原actor的0.30预算。对保存的2020年248日、1240锚点，候选与匹配风险对照仅8日的可行动集合不同；候选与同模型置零相比仅5日、9锚点不同，低于既有20日行动支持门。此处停止，不因5日事后结果降低预算或改变阈值。见 `risk_budget_feasibility_preflight.json`；这不否认其他动作相关经济机制。
